In [1]:
import poolparty as pp

pp.reset()

# Mark changes by changing case of affected bases
pp.set_default('mark_changes',True)

# Define component sequences 
wt_seq  = 'ATCGATCGATATCGATCGAT'        # 10 nt wild-type
ad5_seq = 'GAAC'                        # 3 nt 5' adapter
ad3_seq = 'TGGANNNNNCCAG'               # 3 + 5N barcode + 3

# Create pool of randomly mutagenized wt_seq sequences
mut_pool = pp.mutagenize(wt_seq, num_mutations=3)

# Create pool of 3' adaptors with random barcodes
ad3_pool = pp.from_iupac_motif(ad3_seq, mode='random')

# Join into final oligo pool
oligo_pool = pp.join([ad5_seq, mut_pool, ad3_pool], spacer_str=' ')

# Sample 5 sequences
df = oligo_pool.generate_library(num_seqs=5, seed=42)
print(df['seq'])

0    GAAC AgCGATCGATATCatTCGAT TGGAatatgCCAG
1    GAAC AgCGATCGATtTCGATCGAc TGGAagaatCCAG
2    GAAC ATCGATCGtTtTCGAcCGAT TGGAcgctaCCAG
3    GAAC ATCGATCGcTgTCGtTCGAT TGGAacaacCCAG
4    GAAC ATCGAgCGATATCGAgCGgT TGGAacaccCCAG
Name: seq, dtype: object


In [2]:
# Change columns of df using a dict to standardize names for convenience
rename_dict = {
    'seq': 'seq',
    'op[2]:from_iupac_motif.key.iupac_state': 'iupac_state',
    'op[1]:mutagenize.key.positions': 'positions',
    'op[1]:mutagenize.key.wt_chars': 'wt_chars',
    'op[1]:mutagenize.key.mut_chars': 'mut_chars',
}
tmp_df = df[rename_dict.keys()].copy().rename(columns=rename_dict)
#tmp_df['seq'] = [s[:10]+'...' for s in tmp_df['seq']]
display(tmp_df)

,seq,iupac_state,positions,wt_chars,mut_chars
0,GAAC AgCGATCGATATCatTCGAT TGGAatatgCCAG,206,"(1, 13, 14)","(T, G, A)","(g, a, t)"
1,GAAC AgCGATCGATtTCGATCGAc TGGAagaatCCAG,131,"(1, 10, 19)","(T, A, T)","(g, t, c)"
2,GAAC ATCGATCGtTtTCGAcCGAT TGGAcgctaCCAG,412,"(8, 10, 15)","(A, A, T)","(t, t, c)"
3,GAAC ATCGATCGcTgTCGtTCGAT TGGAacaacCCAG,65,"(8, 10, 14)","(A, A, A)","(c, g, t)"
4,GAAC ATCGAgCGATATCGAgCGgT TGGAacaccCCAG,69,"(5, 15, 18)","(T, T, A)","(g, g, g)"


In [3]:
oligo_pool.print_tree()

pool[4] (pool, n=1)
└── op[4]:join [op=join, mode=fixed, n=1]
    ├── pool[3] (pool, n=1)
    │   └── op[3]:from_seq [op=from_seq, mode=fixed, n=1]
    ├── pool[1] (pool, n=1)
    │   └── op[1]:mutagenize [op=mutagenize, mode=random, n=1]
    │       └── pool[0] (pool, n=1)
    │           └── op[0]:from_seq [op=from_seq, mode=fixed, n=1]
    └── pool[2] (pool, n=1)
        └── op[2]:from_iupac_motif [op=from_iupac_motif, mode=random, n=1]


In [8]:
# Create pool of two insert variants (2 states)
ins_pool = pp.from_seqs(['AAAAA', 'TTTTT'], mode='sequential')

# Replacement scan: 2 inserts x 4 positions = 8 states (Cartesian product)
rep_pool = pp.replacement_scan(
    wt_seq, ins_pool, 
    positions=slice(None,None,5),  
    mode='sequential'
)

# Deletion scan: 4 positions = 4 states
del_pool = pp.deletion_scan(
    wt_seq, 
    deletion_length=5,
    positions=slice(None,None,5),
    mode='sequential'
)

# Stack the two scans: 8 + 4 = 12 states (union)
combo_pool = pp.stack([rep_pool, del_pool])

# Join with adapters (reusing ad5_seq and ad3_pool from Example 1)
oligo_pool_2 = pp.join([ad5_seq, combo_pool, ad3_pool], spacer_str=' ')

# Enumerate all 12 states exactly once
df = oligo_pool_2.generate_library(num_cycles=1, seed=42)
print(df['seq'])

0     GAAC aaaaaTCGATATCGATCGAT TGGAaccgtCCAG
1     GAAC tttttTCGATATCGATCGAT TGGAtacgaCCAG
2     GAAC ATCGAaaaaaATCGATCGAT TGGAggctgCCAG
3     GAAC ATCGAtttttATCGATCGAT TGGActaacCCAG
4     GAAC ATCGATCGATaaaaaTCGAT TGGAcgtgtCCAG
5     GAAC ATCGATCGATtttttTCGAT TGGAtcgttCCAG
6     GAAC ATCGATCGATATCGAaaaaa TGGAaccgaCCAG
7     GAAC ATCGATCGATATCGAttttt TGGAgtaggCCAG
8     GAAC -----TCGATATCGATCGAT TGGAatatgCCAG
9     GAAC ATCGA-----ATCGATCGAT TGGAacgaaCCAG
10    GAAC ATCGATCGAT-----TCGAT TGGAgacgtCCAG
11    GAAC ATCGATCGATATCGA----- TGGAttgctCCAG
Name: seq, dtype: object


In [ ]:
oligo_pool_2.print_tree()